In [1]:
import os
from pathlib import Path
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer
from diffusers import (
    StableDiffusionPipeline,
    DDPMScheduler,
    UNet2DConditionModel,
    AutoencoderKL
)
from diffusers.models.attention_processor import (
    LoRAAttnProcessor
)
from diffusers.loaders import AttnProcsLayers
import matplotlib.pyplot as plt

In [2]:
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
DATASET_DIR = "./photos"
OUTPUT_DIR = "./lora_output"
TOKEN = "sks_person"
PROMPT = f"photo of {TOKEN}"
IMAGE_SIZE = 512
BATCH_SIZE = 1
EPOCHS = 40
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


In [3]:
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
class PersonDataset(Dataset):
    def __init__(self, image_dir, tokenizer):
        self.image_paths = list(Path(image_dir).glob("*"))
        self.tokenizer = tokenizer
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        image = Image.open(
            self.image_paths[idx]
        ).convert("RGB")
        image = self.transform(image)
        tokens = self.tokenizer(
            PROMPT,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt"
        )
        return {
            "pixel_values": image,
            "input_ids": tokens.input_ids[0]
        }

In [5]:
tokenizer = CLIPTokenizer.from_pretrained(
    MODEL_NAME,
    subfolder="tokenizer"
)

In [6]:
vae = AutoencoderKL.from_pretrained(
    MODEL_NAME,
    subfolder="vae"
).to(DEVICE).float()

unet = UNet2DConditionModel.from_pretrained(
    MODEL_NAME,
    subfolder="unet"
).to(DEVICE).float()

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME
).to(DEVICE)

pipe.unet = pipe.unet.float()

pipe.vae = pipe.vae.float()

pipe.text_encoder = pipe.text_encoder.float()

text_encoder = pipe.text_encoder

noise_scheduler = DDPMScheduler.from_pretrained(
    MODEL_NAME,
    subfolder="scheduler"
)

C:\Users\wiad_\PycharmProjects\ItmoCv\.venv\lib\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

In [7]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "to_q",
        "to_k",
        "to_v",
        "to_out.0"
    ],
    lora_dropout=0.1,
    bias="none"
)
unet.add_adapter(lora_config)
print("LoRA initialized")

LoRA initialized


In [8]:
dataset = PersonDataset(
    DATASET_DIR,
    tokenizer
)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [9]:
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=LEARNING_RATE
)

In [10]:
unet = unet.float()
vae = vae.float()
text_encoder = text_encoder.float()
unet.train()
for epoch in range(EPOCHS):
    progress_bar = tqdm(dataloader)
    for batch in progress_bar:
        pixel_values = batch["pixel_values"].to(DEVICE).float()
        input_ids = batch["input_ids"].to(DEVICE)
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents.float()
            latents = latents * 0.18215
        noise = torch.randn_like(latents).float()
        timesteps = torch.randint(0,noise_scheduler.config.num_train_timesteps,(latents.shape[0],),device=DEVICE).long()
        noisy_latents = noise_scheduler.add_noise(
            latents,noise,timesteps).float()
        encoder_hidden_states = text_encoder(input_ids)[0].float()
        noise_pred = unet(
            noisy_latents,timesteps,encoder_hidden_states).sample.float()
        loss = F.mse_loss(
            noise_pred,noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        progress_bar.set_description(
            f"Epoch {epoch+1} | Loss: {loss.item():.4f}"
        )

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

...

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
import os
save_path = "./lora_output/epoch_20"
os.makedirs(save_path, exist_ok=True)
unet.save_attn_procs(save_path)
print("Epoch 20 saved")

Epoch 20 saved


C:\Users\wiad_\PycharmProjects\ItmoCv\.venv\lib\site-packages\diffusers\loaders\unet.py:484: FutureWarning: `save_attn_procs` is deprecated and will be removed in version 0.40.0. Using the `save_attn_procs()` method has been deprecated and will be removed in a future version. Please use `save_lora_adapter()`.
  deprecate("save_attn_procs", "0.40.0", deprecation_message)


In [17]:
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME
).to(DEVICE)

pipe.unet.float()
pipe.vae.float()
pipe.text_encoder.float()

pipe.load_lora_weights(
    "./lora_output/epoch_20"
)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

No LoRA keys associated to UNet2DConditionModel found with the prefix='unet'. This is safe to ignore if LoRA state dict didn't originally have any UNet2DConditionModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


In [18]:
prompts = [

    "portrait photo of sks_person, cyberpunk style, neon lights, futuristic city, realistic photo, high quality",

    "portrait photo of sks_person made of metal, chrome skin, realistic photo, high quality",

    "portrait photo of sks_person man in a forest, realistic photo, high quality",

    "portrait photo of sks_person man in a city, realistic photo, high quality",

    "portrait photo of sks_person man on a beach, realistic photo, high quality",

    "portrait photo of sks_person futuristic sci-fi armor, realistic photo, high quality"
]

In [19]:
generated_images = []
for i, prompt in enumerate(prompts):
    print(f"Generating image {i+1}")
    image = pipe(
        prompt,num_inference_steps=30,guidance_scale=7.5).images[0]
    generated_images.append(image)
    image.save(f"result_{i}.png")

Generating image 1


  0%|          | 0/30 [00:00<?, ?it/s]

Generating image 2


  0%|          | 0/30 [00:00<?, ?it/s]

Generating image 3


...

  0%|          | 0/30 [00:00<?, ?it/s]

Generating image 5


  0%|          | 0/30 [00:00<?, ?it/s]

Generating image 6


  0%|          | 0/30 [00:00<?, ?it/s]